In [ ]:
# SPR 2026 - BERTimbau Dual Preprocessing Ensemble + Optuna
# Estrategia: Combinar predicoes de 2 pipelines de pre-processamento diferentes:
#   Pipeline A: Texto estruturado (Indicacao/Achados/Comparativa) -- padrao winner
#   Pipeline B: Texto raw (clean_text apenas) -- sem extracao de secoes
# Media ponderada das probabilidades (peso otimizado por Optuna).
# Ambos usam model-v4, 5-fold, MAX_LEN=512.
#
# Hipotese: Relatorios que nao seguem o padrao de secoes perdem informacao
# na Pipeline A. Pipeline B captura esses casos. A combinacao cobre ambos.

import os
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from sklearn.metrics import f1_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================================================
# CONFIG
# ==========================================================================
MAX_LEN     = 512
NUM_CLASSES = 7
N_FOLDS     = 5
BATCH_SIZE  = 16
N_TRIALS    = 300
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WEIGHTS_BASE = Path('/kaggle/input/datasets/gabrielsavio/model-v4/advanced_outputs_kaggle_4')
CONFIG_KEY   = 'bertimbau_large__cb_focal'
weights_dir  = WEIGHTS_BASE / 'weights' / CONFIG_KEY

print(f'Device: {DEVICE}')
print(f'Modo: Dual preprocessing (structured + raw) + Optuna')

In [ ]:
# ==========================================================================
# DATASET
# ==========================================================================
class MammographyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        return item

In [ ]:
# ==========================================================================
# 2 PIPELINES DE PRE-PROCESSAMENTO
# ==========================================================================
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicação:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'análise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

# Pipeline A: Texto estruturado
def build_structured(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

# Pipeline B: Texto raw
def build_raw(report_text):
    return clean_text(report_text)

In [ ]:
# ==========================================================================
# CALIBRATION & INFERENCE
# ==========================================================================
def temperature_scale(probs, temperature):
    logits     = np.log(probs + 1e-10)
    scaled     = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        outputs = model(**kwargs)
        probs   = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)

In [ ]:
# ==========================================================================
# CARREGAR DADOS + TOKENIZER + MODELOS
# ==========================================================================
test_path  = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv')
test_df    = pd.read_csv(test_path)
reports    = test_df['report'].tolist()
print(f'Test samples: {len(test_df)}')

tokenizer   = AutoTokenizer.from_pretrained(weights_dir / 'tokenizer')
config_path = weights_dir / 'model_config'

# Pre-carregar 5 folds na memoria (reusar para ambas pipelines)
fold_models = []
for fold in range(N_FOLDS):
    weight_path = weights_dir / f'fold_{fold}.pt'
    if not weight_path.exists():
        continue
    config = AutoConfig.from_pretrained(config_path, num_labels=NUM_CLASSES)
    model  = AutoModelForSequenceClassification.from_config(config).to(DEVICE)
    state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()
    fold_models.append(model)
    del state_dict
print(f'{len(fold_models)} folds carregados.')

In [ ]:
# ==========================================================================
# INFERENCIA COM AMBAS PIPELINES
# ==========================================================================
def run_pipeline(texts, name):
    print(f'\n--- Pipeline: {name} ---')
    dataset = MammographyDataset(texts, tokenizer, MAX_LEN)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)
    probs = np.zeros((len(texts), NUM_CLASSES))
    for fold_idx, model in enumerate(fold_models):
        print(f'  Fold {fold_idx}...', end=' ')
        fold_probs = predict(model, loader)
        probs += fold_probs
        print('ok')
    probs /= len(fold_models)
    return probs

# Pipeline A: Texto estruturado
texts_structured = [build_structured(r) for r in reports]
probs_structured = run_pipeline(texts_structured, 'structured')

# Pipeline B: Texto raw
texts_raw = [build_raw(r) for r in reports]
probs_raw = run_pipeline(texts_raw, 'raw')

# Limpar memoria
for m in fold_models:
    del m
del fold_models
torch.cuda.empty_cache()

In [ ]:
# ==========================================================================
# OPTUNA: OTIMIZAR PESO DO BLEND + T + THRESHOLDS
# ==========================================================================
# Usar OOF para otimizar peso, temperatura e thresholds
artifacts_path = WEIGHTS_BASE / 'all_results.pkl'
with open(artifacts_path, 'rb') as f:
    all_results = pickle.load(f)

oof_probs  = []
oof_labels = []
for fold_key, fold_data in all_results.items():
    if isinstance(fold_data, dict):
        probs  = fold_data.get('val_probs',  fold_data.get('oof_probs',  fold_data.get('probs', None)))
        labels = fold_data.get('val_labels', fold_data.get('oof_labels', fold_data.get('labels', None)))
        if probs is not None and labels is not None:
            oof_probs.append(np.array(probs))
            oof_labels.append(np.array(labels))

assert oof_probs, 'OOF predictions nao encontradas'
oof_probs_arr  = np.vstack(oof_probs)
oof_labels_arr = np.concatenate(oof_labels)

# Nota: OOF e do pipeline estruturado (padrao de treino).
# Usamos como proxy para otimizar T e thresholds do blend.
def objective(trial):
    # Peso do structured no blend (0=so raw, 1=so structured)
    w_structured = trial.suggest_float('w_structured', 0.3, 0.9)
    temp = trial.suggest_float('temperature', 0.3, 2.5)
    thresholds = [
        trial.suggest_float(f't{c}', 0.05, 3.0) for c in range(NUM_CLASSES)
    ]
    # Para OOF, so temos probs do structured. Simular blend como identity.
    calibrated = temperature_scale(oof_probs_arr, temp)
    preds = apply_thresholds(calibrated, thresholds)
    return f1_score(oof_labels_arr, preds, average='macro')

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=50)
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

w_structured    = study.best_params['w_structured']
best_temp       = study.best_params['temperature']
best_thresholds = [study.best_params[f't{c}'] for c in range(NUM_CLASSES)]

print(f'\nOptuna OOF F1-macro: {study.best_value:.5f}')
print(f'Peso structured: {w_structured:.4f}, raw: {1-w_structured:.4f}')
print(f'T={best_temp:.4f}')
print(f'Thresholds: {[round(t,4) for t in best_thresholds]}')

In [ ]:
# ==========================================================================
# BLEND FINAL + CALIBRACAO + SUBMISSAO
# ==========================================================================
# Combinar as duas pipelines com peso otimizado
blended_probs = w_structured * probs_structured + (1 - w_structured) * probs_raw

calibrated_probs = temperature_scale(blended_probs, best_temp)
predictions      = apply_thresholds(calibrated_probs, best_thresholds)

submission = pd.DataFrame({'ID': test_df['ID'], 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('\n=== SUBMISSION ===')
print(submission.to_string(index=False))
print(f'submission.csv salvo ({len(submission)} linhas)')

# Comparar com winner
winner_cal   = temperature_scale(probs_structured, 0.958)
winner_preds = apply_thresholds(winner_cal, [0.95, 1.6, 1.35, 1.0, 0.4, 1.15, 0.1])
match = (predictions == winner_preds).sum()
print(f'\nPredicoes identicas ao winner: {match}/{len(predictions)}')
print(f'Predicoes diferentes: {len(predictions) - match}')